In [1]:
!pip install numpy pandas matplotlib scikit-learn nltk spacy emoji gensim imbalanced-learn torch torchvision torchaudio transformers


  Installing build dependencies: started
  Installing build dependencies: still running...
  Installing build dependencies: finished with status 'error'
  Complete output from command "c:\users\gladish t\appdata\local\programs\python\python37\python.exe" "c:\users\gladish t\appdata\local\programs\python\python37\lib\site-packages\pip" install --ignore-installed --no-user --prefix C:\Users\GLADIS~1\AppData\Local\Temp\pip-build-env-hehad075\overlay --no-warn-script-location --no-binary :none: --only-binary :none: -i https://pypi.org/simple -- setuptools cython>=0.25,<3.0 cymem>=2.0.2,<2.1.0 preshed>=3.0.2,<3.1.0 murmurhash>=0.28.0,<1.1.0 thinc>=8.3.0,<8.4.0 "numpy>=2.0.0,<2.1.0; python_version < '3.9'" "numpy>=2.0.0,<2.1.0; python_version >= '3.9'":
  Ignoring numpy: markers 'python_version >= "3.9"' don't match your environment
    Installing build dependencies: started
    Installing build dependencies: finished with status 'done'
    Getting requirements to build wheel: started
    Ge

Command ""c:\users\gladish t\appdata\local\programs\python\python37\python.exe" "c:\users\gladish t\appdata\local\programs\python\python37\lib\site-packages\pip" install --ignore-installed --no-user --prefix C:\Users\GLADIS~1\AppData\Local\Temp\pip-build-env-hehad075\overlay --no-warn-script-location --no-binary :none: --only-binary :none: -i https://pypi.org/simple -- setuptools cython>=0.25,<3.0 cymem>=2.0.2,<2.1.0 preshed>=3.0.2,<3.1.0 murmurhash>=0.28.0,<1.1.0 thinc>=8.3.0,<8.4.0 "numpy>=2.0.0,<2.1.0; python_version < '3.9'" "numpy>=2.0.0,<2.1.0; python_version >= '3.9'"" failed with error code 1 in None
You are using pip version 19.0.3, however version 24.0 is available.
You should consider upgrading via the 'python -m pip install --upgrade pip' command.


In [ ]:

import os, re, math, json, random, string, gc, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Text & NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import emoji

# Topic modeling
import gensim
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Deep Learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())



PyTorch: 2.5.1+cu118 | CUDA: True


In [ ]:

DATA_PATH = "data/Mental-Health-Twitter.csv" 
df = pd.read_csv(DATA_PATH)
print("Original Columns:", df.columns.tolist())

# Auto-detect text column
if 'post_text' in df.columns:
    df.rename(columns={'post_text': 'text'}, inplace=True)
elif 'tweet' in df.columns:
    df.rename(columns={'tweet': 'text'}, inplace=True)
elif 'clean_text' in df.columns:
    df.rename(columns={'clean_text': 'text'}, inplace=True)

print("Renamed Columns:", df.columns.tolist())
print("Data shape:", df.shape)
df.head()


Original Columns: ['Unnamed: 0', 'post_id', 'post_created', 'post_text', 'user_id', 'followers', 'friends', 'favourites', 'statuses', 'retweets', 'label']
Renamed Columns: ['Unnamed: 0', 'post_id', 'post_created', 'text', 'user_id', 'followers', 'friends', 'favourites', 'statuses', 'retweets', 'label']
Data shape: (20000, 11)


,Unnamed: 0,post_id,post_created,text,user_id,followers,friends,favourites,statuses,retweets,label
0,0,637894677824413696,Sun Aug 30 07:48:37 +0000 2015,It's just over 2 years since I was diagnosed w...,1013187241,84,211,251,837,0,1
1,1,637890384576778240,Sun Aug 30 07:31:33 +0000 2015,"It's Sunday, I need a break, so I'm planning t...",1013187241,84,211,251,837,1,1
2,2,637749345908051968,Sat Aug 29 22:11:07 +0000 2015,Awake but tired. I need to sleep but my brain ...,1013187241,84,211,251,837,0,1
3,3,637696421077123073,Sat Aug 29 18:40:49 +0000 2015,RT @SewHQ: #Retro bears make perfect gifts and...,1013187241,84,211,251,837,2,1
4,4,637696327485366272,Sat Aug 29 18:40:26 +0000 2015,It’s hard to say whether packing lists are mak...,1013187241,84,211,251,837,1,1


In [12]:

print(df['label'].value_counts(dropna=False))
df['text'].head(10).tolist()


label
1    10000
0    10000
Name: count, dtype: int64


["It's just over 2 years since I was diagnosed with #anxiety and #depression. Today I'm taking a moment to reflect on how far I've come since.",
 "It's Sunday, I need a break, so I'm planning to spend as little time as possible on the #A14...",
 'Awake but tired. I need to sleep but my brain has other ideas...',
 "RT @SewHQ: #Retro bears make perfect gifts and are great for beginners too! Get stitching with October's Sew on sale NOW! #yay http://t.co/…",
 'It’s hard to say whether packing lists are making life easier or just reinforcing how much still needs doing... #movinghouse #anxiety',
 'Making packing lists is my new hobby... #movinghouse',
 'At what point does keeping stuff for nostalgic reasons cross the line into plain old hoarding...? #movinghouse',
 'Currently in the finding-boxes-of-random-shit packing phase. I think I’m a closet hoarder...',
 "Can't be bothered to cook, take away on the way 😁👍🏼 #lazy",
 'RT @itventsnews: ITV releases promo video for the final series of Down

In [ ]:

nltk.download('stopwords')
nltk.download('wordnet')

STOPWORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Basic cleaner
URL_PATTERN = re.compile(r"http\S+|www\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#\w+")
MULTISPACE_PATTERN = re.compile(r"\s+")

def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = URL_PATTERN.sub(" ", s)
    s = MENTION_PATTERN.sub(" ", s)
    s = HASHTAG_PATTERN.sub(" ", s)
    s = emoji.replace_emoji(s, replace=" ")  # keep emoji removed here; we'll extract counts separately before removal
    s = re.sub(r"[^a-z\s]", " ", s)
    tokens = [lemmatizer.lemmatize(tok) for tok in s.split() if tok not in STOPWORDS and len(tok) > 2]
    s = " ".join(tokens)
    s = MULTISPACE_PATTERN.sub(" ", s).strip()
    return s

# Preserve original text for emoji counts; create a cleaned copy
df['text_raw'] = df['text'].astype(str)
df['text_clean'] = df['text_raw'].apply(clean_text)

print("Sample cleaned text:")
df[['text_raw','text_clean']].head()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\karthick\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\karthick\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Sample cleaned text:


,text_raw,text_clean
0,It's just over 2 years since I was diagnosed w...,year since diagnosed today taking moment refle...
1,"It's Sunday, I need a break, so I'm planning t...",sunday need break planning spend little time p...
2,Awake but tired. I need to sleep but my brain ...,awake tired need sleep brain idea
3,RT @SewHQ: #Retro bears make perfect gifts and...,bear make perfect gift great beginner get stit...
4,It’s hard to say whether packing lists are mak...,hard say whether packing list making life easi...


In [14]:

# Simple emoji sentiment lexicon.
POSITIVE_EMOJIS = set(list("😀😃😄😁😆😊🙂😍😘😺👍💪❤️💕✨🎉😌😎😻🥳🤗🤩☺️"))
NEGATIVE_EMOJIS = set(list("😞😟😔😢😭😠😡😣😖😫😩💔😞😓😥😪😿"))
NEUTRAL_EMOJIS  = set(list("😐🤔😶🙃🤨😑"))

def emoji_sentiment_counts(text: str):
    pos = neg = neu = 0
    if not isinstance(text, str):
        return pd.Series([0,0,0])
    for ch in text:
        if ch in POSITIVE_EMOJIS: pos += 1
        elif ch in NEGATIVE_EMOJIS: neg += 1
        elif ch in NEUTRAL_EMOJIS:  neu += 1
    return pd.Series([pos, neg, neu])

df[['emoji_pos','emoji_neg','emoji_neu']] = df['text_raw'].apply(emoji_sentiment_counts)

print(df[['text_raw','emoji_pos','emoji_neg','emoji_neu']].head())


                                            text_raw  emoji_pos  emoji_neg  \
0  It's just over 2 years since I was diagnosed w...          0          0   
1  It's Sunday, I need a break, so I'm planning t...          0          0   
2  Awake but tired. I need to sleep but my brain ...          0          0   
3  RT @SewHQ: #Retro bears make perfect gifts and...          0          0   
4  It’s hard to say whether packing lists are mak...          0          0   

   emoji_neu  
0          0  
1          0  
2          0  
3          0  
4          0  


In [15]:

# Build dictionary & corpus for LDA using cleaned text
texts = [t.split() for t in df['text_clean'].fillna("").tolist()]
dictionary = Dictionary(texts)
# Filter extremes: remove very rare/common tokens
dictionary.filter_extremes(no_below=2, no_above=0.5, keep_n=5000)

corpus = [dictionary.doc2bow(toks) for toks in texts]

# Train a small LDA (adjust num_topics to your dataset size)
NUM_TOPICS = 5
if len(dictionary) > 0 and len(corpus) > 0:
    lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=NUM_TOPICS, random_state=SEED, passes=5, iterations=200)
    # Assign dominant topic per tweet
    def dominant_topic(bow):
        dist = lda.get_document_topics(bow, minimum_probability=0.0)
        # dist is list of (topic_id, prob)
        if not dist:
            return -1
        return max(dist, key=lambda x: x[1])[0]
    df['topic_id'] = [dominant_topic(bow) for bow in corpus]
else:
    df['topic_id'] = 0  # fallback for tiny demo

print("LDA topics (top words):")
if 'lda' in locals():
    for k in range(NUM_TOPICS):
        print(f"Topic {k}:", lda.print_topic(k, topn=8))

df[['text_clean','topic_id']].head()


LDA topics (top words):
Topic 0: 0.024*"love" + 0.015*"best" + 0.015*"zayn" + 0.014*"one" + 0.011*"want" + 0.010*"good" + 0.010*"get" + 0.010*"video"
Topic 1: 0.025*"trump" + 0.019*"thanks" + 0.012*"fuck" + 0.010*"real" + 0.010*"see" + 0.009*"like" + 0.009*"tweet" + 0.008*"amp"
Topic 2: 0.011*"man" + 0.010*"like" + 0.010*"let" + 0.009*"amp" + 0.009*"take" + 0.009*"gonna" + 0.008*"girl" + 0.008*"one"
Topic 3: 0.054*"user" + 0.029*"say" + 0.028*"thank" + 0.026*"twitter" + 0.022*"following" + 0.020*"hello" + 0.016*"know" + 0.014*"people"
Topic 4: 0.035*"yong" + 0.028*"hey" + 0.025*"follow" + 0.015*"wearepayting" + 0.015*"depression" + 0.015*"foryong" + 0.011*"paytforluckysun" + 0.009*"yeah"


,text_clean,topic_id
0,year since diagnosed today taking moment refle...,3
1,sunday need break planning spend little time p...,2
2,awake tired need sleep brain idea,1
3,bear make perfect gift great beginner get stit...,0
4,hard say whether packing list making life easi...,3


In [16]:

X = df[['text_clean','emoji_pos','emoji_neg','emoji_neu','topic_id']].copy()
y = df['label'].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y if len(np.unique(y))>1 else None)

print("Train size:", len(X_train), " Test size:", len(X_test))


Train size: 16000  Test size: 4000


## Classical ML Pipelines (TF-IDF + Emoji + Topic)

In [17]:

# ColumnTransformer: TF-IDF on text; numeric on emoji counts; one-hot on topic_id
text_features = 'text_clean'
numeric_features = ['emoji_pos','emoji_neg','emoji_neu']
cat_features = ['topic_id']

preprocess = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=2), text_features),
        ('num', StandardScaler(with_mean=False), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ],
    remainder='drop'
)

# Logistic Regression pipeline with oversampling (optional if class imbalance)
lr_pipeline = ImbPipeline(steps=[
    ('preprocess', preprocess),
    ('sampler', RandomOverSampler(random_state=SEED)),
    ('clf', LogisticRegression(max_iter=200, class_weight='balanced'))
])

rf_pipeline = ImbPipeline(steps=[
    ('preprocess', preprocess),
    ('sampler', RandomOverSampler(random_state=SEED)),
    ('clf', RandomForestClassifier(n_estimators=300, random_state=SEED, class_weight='balanced'))
])

def eval_and_report(model, X_tr, y_tr, X_te, y_te, name="Model"):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    proba = None
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X_te)[:,1]
        except:
            proba = None
    acc = accuracy_score(y_te, preds)
    f1 = f1_score(y_te, preds, average='weighted')
    print(f"\n{name} -> Accuracy: {acc:.4f} | F1 (weighted): {f1:.4f}")
    print(classification_report(y_te, preds, digits=4))
    cm = confusion_matrix(y_te, preds)
    print("Confusion Matrix:\n", cm)
    return acc, f1, preds, proba

acc_lr, f1_lr, preds_lr, proba_lr = eval_and_report(lr_pipeline, X_train, y_train, X_test, y_test, name="Logistic Regression")
acc_rf, f1_rf, preds_rf, proba_rf = eval_and_report(rf_pipeline, X_train, y_train, X_test, y_test, name="Random Forest")



Logistic Regression -> Accuracy: 0.7552 | F1 (weighted): 0.7552
              precision    recall  f1-score   support

           0     0.7633    0.7400    0.7515      2000
           1     0.7477    0.7705    0.7589      2000

    accuracy                         0.7552      4000
   macro avg     0.7555    0.7552    0.7552      4000
weighted avg     0.7555    0.7552    0.7552      4000

Confusion Matrix:
 [[1480  520]
 [ 459 1541]]

Random Forest -> Accuracy: 0.7318 | F1 (weighted): 0.7313
              precision    recall  f1-score   support

           0     0.7534    0.6890    0.7198      2000
           1     0.7135    0.7745    0.7427      2000

    accuracy                         0.7318      4000
   macro avg     0.7335    0.7317    0.7313      4000
weighted avg     0.7335    0.7318    0.7313      4000

Confusion Matrix:
 [[1378  622]
 [ 451 1549]]


## LSTM Model (with Tokenizer + Embedding)

In [18]:

from collections import Counter
import itertools

# Build simple word index from cleaned text
def build_vocab(texts, min_freq=2, max_size=20000):
    counter = Counter()
    for t in texts:
        counter.update(t.split())
    # Keep tokens with min frequency
    vocab = [w for w, c in counter.items() if c >= min_freq]
    vocab = vocab[:max_size-2]  # reserve for PAD and UNK
    idx2word = ["<PAD>", "<UNK>"] + vocab
    word2idx = {w:i for i,w in enumerate(idx2word)}
    return word2idx, idx2word

word2idx, idx2word = build_vocab(X_train['text_clean'].tolist(), min_freq=2, max_size=30000)

def encode_text(s, max_len=64):
    toks = s.split()
    ids = [word2idx.get(w, 1) for w in toks]  # 1 is <UNK>
    if len(ids) < max_len:
        ids = ids + [0]*(max_len - len(ids))  # 0 is <PAD>
    else:
        ids = ids[:max_len]
    return np.array(ids, dtype=np.int64)

class TweetDataset(Dataset):
    def __init__(self, df, labels):
        self.ids = np.vstack(df['text_clean'].apply(lambda s: encode_text(s, max_len=64)).values)
        self.emojis = df[['emoji_pos','emoji_neg','emoji_neu']].values.astype(np.float32)
        self.topics = df['topic_id'].values.astype(np.int64)
        self.labels = labels.astype(np.int64)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "ids": torch.tensor(self.ids[idx]),
            "emojis": torch.tensor(self.emojis[idx]),
            "topics": torch.tensor(self.topics[idx]),
            "labels": torch.tensor(self.labels[idx])
        }

train_ds = TweetDataset(X_train, y_train)
test_ds  = TweetDataset(X_test, y_test)

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_topics=10):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.top_emb = nn.Embedding(num_topics+1, 16)  # +1 for safety if topic -1
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim*2 + 3 + 16, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
    def forward(self, ids, emojis, topics):
        x = self.embed(ids)
        _, (h_n, _) = self.lstm(x)
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)  # bi-LSTM last hidden
        topics = torch.clamp(topics, min=0)  # avoid -1
        t = self.top_emb(topics)
        feat = torch.cat([h, emojis, t], dim=1)
        logits = self.fc(feat)
        return logits

BATCH = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_topics = int(df['topic_id'].max()) if 'topic_id' in df.columns else 5
model_lstm = LSTMClassifier(vocab_size=len(idx2word), embed_dim=128, hidden_dim=128, num_topics=num_topics).to(device)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH)

opt = torch.optim.Adam(model_lstm.parameters(), lr=2e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 3  # increase for better performance
for epoch in range(EPOCHS):
    model_lstm.train()
    tot_loss = 0.0
    for batch in train_loader:
        ids = batch['ids'].to(device)
        emojis = batch['emojis'].to(device)
        topics = batch['topics'].to(device)
        labels = batch['labels'].to(device)
        opt.zero_grad()
        logits = model_lstm(ids, emojis, topics)
        loss = criterion(logits, labels)
        loss.backward()
        opt.step()
        tot_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {tot_loss/len(train_loader):.4f}")

# Evaluate
model_lstm.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for batch in test_loader:
        ids = batch['ids'].to(device)
        emojis = batch['emojis'].to(device)
        topics = batch['topics'].to(device)
        labels = batch['labels'].to(device)
        logits = model_lstm(ids, emojis, topics)
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_true.extend(labels.cpu().numpy().tolist())

acc = accuracy_score(all_true, all_preds)
f1 = f1_score(all_true, all_preds, average='weighted')
print(f"LSTM -> Accuracy: {acc:.4f} | F1 (weighted): {f1:.4f}")
print(classification_report(all_true, all_preds, digits=4))


Epoch 1/3 | Train Loss: 0.5894
Epoch 2/3 | Train Loss: 0.4428
Epoch 3/3 | Train Loss: 0.3119
LSTM -> Accuracy: 0.7400 | F1 (weighted): 0.7400
              precision    recall  f1-score   support

           0     0.7449    0.7300    0.7374      2000
           1     0.7353    0.7500    0.7426      2000

    accuracy                         0.7400      4000
   macro avg     0.7401    0.7400    0.7400      4000
weighted avg     0.7401    0.7400    0.7400      4000



##  DistilBERT Fine-tuning

In [19]:

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class HFTextDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(texts, truncation=True, padding=False, max_length=128)
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]))
        return item

train_texts = X_train['text_clean'].tolist()
test_texts  = X_test['text_clean'].tolist()

train_hf = HFTextDataset(train_texts, y_train)
test_hf  = HFTextDataset(test_texts, y_test)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

num_labels = 2
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

args = TrainingArguments(
    output_dir="data/bert_outputs",
    evaluation_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,  # increase to 3-5 for better results
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=False
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_hf,
    eval_dataset=test_hf,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`

##  Visualizations

In [ ]:

def plot_confusion_matrix(cm, classes):
    fig = plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title('Confusion matrix')
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    thresh = cm.max() / 2.
    for i, j in np.ndindex(cm.shape):
        plt.text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment="center")
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    return fig

# Use the LR predictions from earlier for a simple viz (re-train quickly if needed)
_ = lr_pipeline.fit(X_train, y_train)
preds = lr_pipeline.predict(X_test)
cm = confusion_matrix(y_test, preds)
fig = plot_confusion_matrix(cm, classes=["Non-Depressed","Depressed"])
plt.show()


##  Save Artifacts (Models & Preprocessing)

In [ ]:
import os
import joblib
import torch

# Define artifact directory
ART_DIR = "data/artifacts"
os.makedirs(ART_DIR, exist_ok=True)

# ✅ Save classical ML models
joblib.dump(lr_pipeline, os.path.join(ART_DIR, "lr_pipeline.joblib"))
joblib.dump(rf_pipeline, os.path.join(ART_DIR, "rf_pipeline.joblib"))

# ✅ Save LSTM model
torch.save(model_lstm.state_dict(), os.path.join(ART_DIR, "lstm_state_dict.pt"))

# ✅ Save DistilBERT model (only if trainer exists)
if 'trainer' in globals() and 'tokenizer' in globals():
    bert_dir = os.path.join(ART_DIR, "distilbert_model")
    os.makedirs(bert_dir, exist_ok=True)
    trainer.model.save_pretrained(bert_dir)
    tokenizer.save_pretrained(bert_dir)
    print("DistilBERT model saved successfully ✅")
else:
    print("⚠️ DistilBERT model not trained — skipping save step.")

print("✅ All available models saved successfully to:", ART_DIR)


⚠️ DistilBERT model not trained — skipping save step.
✅ All available models saved successfully to: data/artifacts


##  Inference Helper (Single Tweet Prediction)

In [22]:

def predict_single(text: str):
    # Build feature row
    pos, neg, neu = emoji_sentiment_counts(text)
    clean = clean_text(text)
    # LDA topic for this text
    bow = dictionary.doc2bow(clean.split()) if 'dictionary' in globals() else []
    if 'lda' in globals() and len(bow)>0:
        topic = np.argmax([prob for _, prob in lda.get_document_topics(bow, minimum_probability=0.0)])
    else:
        topic = 0
    row = pd.DataFrame([{
        "text_clean": clean,
        "emoji_pos": pos,
        "emoji_neg": neg,
        "emoji_neu": neu,
        "topic_id": topic
    }])
    proba = None
    if hasattr(lr_pipeline, "predict_proba"):
        try:
            proba = lr_pipeline.predict_proba(row)[0,1]
        except:
            proba = None
    pred = lr_pipeline.predict(row)[0]
    return pred, proba

example = "I feel empty and anxious... nothing feels right 😞"
p, s = predict_single(example)
print("Prediction:", "Depressed" if p==1 else "Non-Depressed", "| Score:", s)


Prediction: Depressed | Score: 0.6496002048741724
